In [1]:
from pathlib import Path
import dataclasses
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").is_dir() and (candidate / "lane_graph").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from waymax import config as waymax_config
from data.scenario_loader import load_scenario_state_fast
from lane_graph.lane_graph_utils import LaneGraphLoader

# 실제 데이터 위치로만 바꾸면 됩니다.
tfrecord_path = "/zfsauton/scratch/eshau/womd/tf_example/training/training_tfexample.tfrecord-00999-of-01000"
lane_graph_dir = "/zfsauton/scratch/mineuih/waymax_rs/lane_graphs"
scenario_index = 0

ds_cfg = dataclasses.replace(
    waymax_config.WOD_1_3_1_TRAINING,
    path=tfrecord_path,
    batch_dims=(1,),
    shuffle_seed=0,
)

sim_state = load_scenario_state_fast(ds_cfg, scenario_index)
lane_graph_loader = LaneGraphLoader(lane_graph_dir)
lane_graph = lane_graph_loader.get_lane_graph_for_scenario(tfrecord_path, scenario_index)

print("Loaded scenario_index:", scenario_index)
print("sim_state log_trajectory.xy shape:", sim_state.log_trajectory.xy.shape)

if lane_graph is None:
    print("lane_graph: not found for this tfrecord/scenario_index")
else:
    print("lane_graph scenario_id:", lane_graph.scenario_id)
    print("lane_graph nodes:", lane_graph.nodes_xyz.shape[0])
    print("lane_graph lanes:", len(lane_graph.lane_id_to_node_range))

I0000 00:00:1779683626.219499   58264 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1779683635.396711   58264 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
I0000 00:00:1779683635.465981   58524 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Loaded scenario_index: 0
sim_state log_trajectory.xy shape: (1, 128, 91, 2)
lane_graph scenario_id: a6a5089f8a308f2b
lane_graph nodes: 13124
lane_graph lanes: 249


In [2]:
print(lane_graph.lane_ids)
print(sim_state.roadgraph_points.ids[0, :10])
print(sim_state.roadgraph_points.xy[0, :10000] == lane_graph.nodes_xyz[:10000, :2])

[103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120
 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138
 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156
 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174
 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192
 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210
 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228
 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246
 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264
 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281 282
 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 311 312 313
 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331
 332 333 334 335 336 337 338 345 346 347 351 366 367 368 369 370 371 372
 373 374 375 376 377 378 379 380 381 382 383 384 38

In [18]:
print(sim_state.log_traffic_light.state.shape)

(1, 16, 91)


In [16]:
print(lane_graph.left_neighbor_edges[:20])
cnt = 0
for lane_id, node_range in lane_graph.lane_id_to_node_range.items():
    print(f"{lane_id}: {node_range[0]}, {node_range[1]}")
    cnt += 1
    if cnt >= 20:
        break

print(lane_graph.successor_edges[:20])

[[ 93 110]
 [136 153]
 [136 110]
 [259 235]
 [259 153]
 [356 390]
 [356 515]
 [422 390]
 [422 356]
 [422 515]
 [422 554]
 [422 874]
 [554 515]
 [554 874]
 [593 693]
 [593 840]
 [613 942]
 [654 693]
 [654 390]
 [735 654]]
103: 0, 6
104: 6, 14
105: 14, 93
106: 93, 110
107: 110, 136
108: 136, 153
109: 153, 182
110: 182, 235
111: 235, 259
112: 259, 285
113: 285, 350
114: 350, 356
115: 356, 390
116: 390, 422
117: 422, 454
118: 454, 515
119: 515, 554
120: 554, 593
121: 593, 613
122: 613, 630
[[   5  356]
 [   5  422]
 [   5  390]
 [  13  898]
 [  92 7667]
 [  92 7632]
 [ 109  182]
 [ 135  285]
 [ 152  285]
 [ 181   14]
 [ 234 7881]
 [ 234 7863]
 [ 258  182]
 [ 284   14]
 [ 349  554]
 [ 349  593]
 [ 349  515]
 [ 389  749]
 [ 421  454]
 [ 453  642]]
